In [ ]:
import json
from pathlib import Path
import requests
import xarray as xr
import numpy as np
import datetime
import pandas as pd
import matplotlib.pyplot as plt
from erddapy import ERDDAP

### Authentication

You can add your auth in a file `secrets.json` in the home directory of this repo

In [ ]:
oceanops_auth = {'X-OceanOPS-Metadata-ID': 'nnnn',
 'X-OceanOPS-Metadata-Token': '16-char-secret-token'}

if Path('../secrets.json').exists():
    with open('../secrets.json') as f:
        oceanops_auth = json.load(f)
else:
    print("enter your oceanOPS authentication in the dictionary `oceanops_auth` in this cell")

In [ ]:
headers = {
    'Content-Type': 'application/json' ,
    'accept': 'application/json' ,
    **oceanops_auth
}

# Use WMO ID of glider to get all OceanOPS passports info on that platform

1. We enumerate through the potential passport IDs for a given wmo_id until the API returns a 404

In [ ]:
wmo_id = 6801674
platform_serial = 'SEA055'
wigos_ids = [f'0-22000-{num}-{wmo_id}' for num in range(1000)]

In [ ]:
response.content

In [ ]:
response.status_code

In [ ]:
oceanops_info = {}

for wigos_id in wigos_ids:
    _, _, passport_num, _ = wigos_id.split('-')
    url = f'https://www.ocean-ops.org/api/data/platforms/{wigos_id}'
    response = requests.patch(url, headers=headers, json={})
    if response.status_code == 404:
        print(f"Maxed out at {wigos_id} which does not exist. Total number of existing wigos IDs for this platform {wmo_id}: {passport_num}")
        break
    elif response.status_code==200:
        resp_json = response.json()
        oceanops_info[wigos_id] = resp_json
    else:
        print(f"Request returned unexteced status code {response.status_code}")
        print(response.content)


2. We flatten these dictionaries of data into a single pandata DataFrame. This summarises all the data OceanOPS currently has in the passports for this platform

In [ ]:
oceanops_info_flat = {}
for dataset_id, ddict in oceanops_info.items():
    _, _, passport_num, _ = ddict['wigosId'].split('-')
    flat_ddict = {'passport_num': int(passport_num)}
    for key, val in ddict.items():
        if type(val) != dict:
            flat_ddict[key] = val
            continue
        for sub_key, sub_val in val.items():
            flat_ddict[f'{key}_{sub_key}'] = str(sub_val)
    oceanops_info_flat[dataset_id] = flat_ddict
df_oceanops = pd.DataFrame(oceanops_info_flat).T
df_oceanops['deployment_date'] = pd.to_datetime(df_oceanops.deployment_date)
df_oceanops['retrieval_startDate'] = pd.to_datetime(df_oceanops.retrieval_startDate)
df_oceanops['retrieval_endDate'] = pd.to_datetime(df_oceanops.retrieval_endDate)

df_oceanops = df_oceanops.sort_values('deployment_date')
df_oceanops

Let's print a more digestible version of that table

In [ ]:
df_oceanops[['passport_num',  'name',  'deployment_date', 'deployment_latitude', 'deployment_longitude', 'retrieval_startDate', 'retrieval_endDate', 'retrieval_endingCause', ]]


3. Plot temporal extent of passports

Passports with start and end date show as lines, those with only start date are circles, those with only recovery date are squares

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
for i, row in df_oceanops.iterrows():
    if not pd.isnull(row.deployment_date) and not pd.isnull(row.retrieval_startDate):
        ax.axvspan(row.deployment_date, row.retrieval_startDate, alpha=0.4)

        ax.plot((row.deployment_date, row.retrieval_startDate), (row.passport_num, row.passport_num))
        continue
    if not pd.isnull(row.deployment_date):
        ax.scatter((row.deployment_date, ), (row.passport_num))
        continue
    ax.scatter((row.retrieval_startDate, ), (row.passport_num), marker='s')
ax.set(ylabel='passport number');

# Update info on WMO passports

Update the deployment and recovery dates of missions and request passports for missions that do not have them

1. loop through the ERDDAP table of missions chronologically and make a table of the data we have on each mission

In [ ]:
e = ERDDAP(
    server="https://erddap.observations.voiceoftheocean.org/erddap",
    protocol="tabledap",
)
e.response = "csv"
e.dataset_id = "allDatasets"
df_datasets = e.to_pandas()
df_datasets = df_datasets[df_datasets.datasetID.str.contains("OG_nrt")]
df_ids = df_datasets.datasetID.str[7:].str.split('_',expand=True)
df_datasets['platform_serial'] = df_ids[0]
df_datasets['mission'] = df_ids[1].str[1:].astype(int)
df_platform = df_datasets[df_datasets.platform_serial==platform_serial]
df_platform = df_platform.rename({'minTime (UTC)': 'deployment_date', 'maxTime (UTC)': 'recovery_date'}, axis=1)
vars =['datasetID', 'title',  'summary', 'platform_serial', 'mission', 'deployment_date', 'recovery_date']
df_platform['deployment_date'] = pd.to_datetime(df_platform.deployment_date)
df_platform['recovery_date'] = pd.to_datetime(df_platform.recovery_date)
df_platform = df_platform[vars]
df_platform = df_platform.sort_values('deployment_date')
df_platform['gtsId'] = 6801506
print(f"Found data for {len(df_platform)} deployments on ERDDAP")
df_platform

2. Use the deployment date to match missions in our table to existing passports

In [ ]:
df_platform['wigosId'] = ''
matches = 0
for i, row in df_platform.iterrows():
    deploy_diff = df_oceanops.deployment_date - row.deployment_date
    time_match = np.abs(deploy_diff) < np.timedelta64(3, 'D')
    if time_match.any():
        matching_passports = df_oceanops[time_match]
        if len(matching_passports == 1):
            matches += 1
            df_platform.loc[i, 'wigosId'] = matching_passports['wigosId'].values[0]
        else:
            print("Error! More than one match detected")
print(f'Matched {matches} datasets')

4. Check for datasets on ERDDAP with no corresponding wigos ID

In [ ]:
datasets_no_wigos = df_platform[df_platform['wigosId']=='']
if not datasets_no_wigos.empty:
    print(f"Failed to find wigos IDs for these datasetson  ERDDAP:")
datasets_no_wigos[['datasetID',  'deployment_date', 'recovery_date']]

5. Check for datasets on oceanOPS with no corresponding dataset on our side

In [ ]:
unmatched_wigos = set(df_oceanops.index).difference(set(df_platform.wigosId))
if unmatched_wigos:
    print("No corresponding dataset found on ERDDAP for assigned wigos passports:")
df_oceanops[df_oceanops.wigosId.isin(unmatched_wigos)][['wigosId', 'deployment_date', 'retrieval_startDate']]

3. Merge the OceanOPS table and our table to compare information

In [ ]:
dfa = df_platform[['datasetID', 'mission', 'title', 'deployment_date', 'recovery_date', 'wigosId', 'gtsId', 'platform_serial']]

dfb =df_oceanops[['passport_num', 'wigosId', 'name','deployment_date', 'identification_internalId', 'retrieval_startDate','retrieval_endDate', 'retrieval_endingCause' ]]
df = pd.merge(dfa, dfb, left_on='wigosId', right_on='wigosId', suffixes=('_voto', '_ops'), how='left')
df[df.wigosId!= '']

Plot temporal extents of OceanOPS and VOTO deployments

In [ ]:
fig, ax = plt.subplots(2, 1, sharex='col', figsize=(14, 10))
ax = ax.ravel()
for i, row in df_oceanops.iterrows():
    if not pd.isnull(row.deployment_date) and not pd.isnull(row.retrieval_startDate):
        color='C1' if row.wigosId in unmatched_wigos else 'C0'
        ax[0].axvspan(row.deployment_date, row.retrieval_startDate, alpha=0.4, color=color)

        ax[0].plot((row.deployment_date, row.retrieval_startDate), (row.passport_num, row.passport_num))
        continue
    if not pd.isnull(row.deployment_date):
        ax[0].scatter((row.deployment_date, ), (row.passport_num))
        continue
    ax[0].scatter((row.retrieval_startDate, ), (row.passport_num), marker='s')
ax[0].set(ylabel='passport number');

for i, row in df_platform.iterrows():
    color='C1' if row.wigosId == '' else 'C0'

    ax[1].axvspan(row.deployment_date, row.recovery_date, alpha=0.4, color=color)

    ax[1].plot((row.deployment_date, row.recovery_date), (row.mission, row.mission))
ax[1].set(ylabel='mission ID');

4. Go back over the table, updating recovery dates for missions that do not have them

In [ ]:
for i, row in df.iterrows():
    if row.wigosId:
        if pd.isnull(row.retrieval_endingCause):
            json_data =  {
            "retrieval": {
                "endingCause": {
          "code": "voluntary-recovery-of-the-platform"
                        }
                    }
            }

            url = f'https://www.ocean-ops.org/api/data/platforms/{row.wigosId}'
            response = requests.patch(url, headers=headers, json=json_data)
            resp = response.json()
            print(resp)

        if not pd.isnull(row.retrieval_startDate) and not pd.isnull(row.retrieval_endDate):
            continue
        json_data =   {
            "retrieval": {
                "startDate": row.recovery_date.isoformat().replace('+00:00', 'Z'),
                "endDate": row.recovery_date.isoformat().replace('+00:00', 'Z')}
                      }
        url = f'https://www.ocean-ops.org/api/data/platforms/{row.wigosId}'
        response = requests.patch(url, headers=headers, json=json_data)
        resp_json = response.json()
        print(resp_json)


5. Request passport IDs for missions that do not have them
6. 
**Note** the API will return a 200 even if it rejects your request to mint a new Passport ID in some circumstances, check the returned json for a 'success' message

In [ ]:
for i, row in df.iterrows():
    if row.wigosId:
        continue
    url = 'https://www.ocean-ops.org/api/data/platforms/getid'

    json_data =   [{
        "program": "VOTO: Ocean Knowledge glider program",
        "model": "SeaExplorer X2",
        "startDate": row.deployment_date_voto.isoformat().replace('+00:00', ''),
        "endDate": row.recovery_date.isoformat().replace('+00:00', ''),
        "gtsId": row.gtsId,
        "closeCurrentMission": "true"
      }]

    response = requests.post(url, headers=headers, json=json_data)
    print("Status", response.status_code)
    resp_json = response.json()[0]
    if not resp_json['success']:
        print("ERROR request denied")
    print(resp_json)

# Generate tables for ERDDAP that oceanOPS will harvest

This is an example pulling from the VOTO ERDDAP. ymmv!

Using the CSV column names from https://www.ocean-ops.org/passports/#/submit

We work through the WIGOS Passport ids that we have, extract metadata from the ERDDAP dataset of each mission, and compile these metadata into two tables, following the example from GOOS.

Note that the network comes from this table https://www.ocean-ops.org/passports/#/code-tables?code=CT-GNW

In [ ]:
passport_dicts = []
sensor_dicts = []
for i, row in df.iterrows():
    e.dataset_id = row.datasetID
    passport_id = row.wigosId
    platform_serial = row.platform_serial
    
    ds = e.to_xarray()

    attrs = ds.attrs
    
    platform_serial = dataset_id.split('_M')[0]
    passport_dict = {'PASSPORT IDENTIFIER': passport_id,
                     'PROGRAM': 'VOTO: Ocean Knowledge glider program',
                     'PLATFORM MODEL': 'SeaExplorer X2',
                     'PLATFORM SERIAL NUMBER': platform_serial,
                     'DEPLOYMENT DATE': str(ds.time.values[~np.isnan(ds.time)][0])[:19] + 'Z',
                     'DEPLOYMENT LATITUDE': ds.latitude.values[~np.isnan(ds.latitude)][0],
                     'DEPLOYMENT LONGITUDE': ds.longitude.values[~np.isnan(ds.longitude)][0],
                     'DEPLOYMENT SHIP': 'Seeker', # tbd
                     'GTS ID': int(attrs['wmo_id']),
                     'TELECOM TYPE': 'iridium',                    
    }
    
    # conditional entries
    conditional_dict = {
        'PRINCIPAL INVESTIGATOR': 'Louise Biddle',
        'AGENCIES': 'VOTO',
        'AGENCY_ROLES': 'operating agency',
        'CONTRIBUTORS': attrs['contributor_name'],
        'CONTRIBUTOR_ROLE': attrs['contributor_role']
        
    }
    end_time = ds.time.values[~np.isnan(ds.time)][-1]
    
    if end_time < np.datetime64(datetime.datetime.now() - datetime.timedelta(days=3)):
        passport_dict['END DATE'] = str(ds.time.values[~np.isnan(ds.time)][-1])[:19] + 'Z'
        passport_dict['ENDING CAUSE'] = 'recovered'
    passport_dict = passport_dict | conditional_dict
    passport_dicts.append(passport_dict)  

    from string import ascii_uppercase as auc
    i = 0
    for var_name in ds.variables:
        variable = ds[var_name]
        attrs = variable.attrs
        if 'sensor' not in attrs.keys():
            continue
        if attrs['sensor'][:6] != 'SENSOR':
            continue
        sensor_dict = ds[attrs['sensor']].attrs
        if 'standard_name' not in attrs.keys():
            continue
        sensor_dict = {
            'PASSPORT IDENTIFIER': passport_id,
            'SHORTCODE': auc[i],
            'SENSOR MODEL': sensor_dict['sensor_model'],
            'VARIABLE': attrs['standard_name'],
            'SERIAL NUMBER': sensor_dict['sensor_serial_number'],
        }
        i += 1
        sensor_dicts.append(sensor_dict)
        

In [ ]:
df_passport = pd.DataFrame(passport_dicts)
df_passport.to_csv('passport.csv', sep=';', index=False)
df_passport

In [ ]:
df_sensor = pd.DataFrame(sensor_dicts)
df_sensor.to_csv('sensor.csv', sep=';', index=False)

df_sensor

In [ ]:
df_oceanops